# Utilities for random search

Example pipeline:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe = Pipeline([
    ("scaler", StandardScaler()), 
    ("pca", PCA(svd_solver="full")),
    ("knn", KNeighborsClassifier(algorithm="brute", n_jobs=1))
])

Example parameters:

In [ ]:
param_dist = {
    "pca__n_components": st.uniform(0.90, 0.10),
    "knn__n_neighbors": st.randint(50, 150),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

randomized search:

In [ ]:
rs = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20, # try 8–20 depending on time
    cv=3, # start with less CV to catch a pattern
    n_jobs=-1, # !!! if n_jobs=-1 then n_jobs in the model is 1
    scoring="accuracy",
    random_state=42,
    verbose=1
)
rs.fit(X_train, y_train)
print("Best params:", rs.best_params_)
print("CV best score:", rs.best_score_)
print("Test acc:", rs.score(X_test, y_test))

display a parameter agains the score:

In [ ]:
def plot_parameter_vs_score(rs, parameter, score):
  
  # Get CV results into a DataFrame
  df = pd.DataFrame(rs.cv_results_)
  
  # Extract relevant columns
  df_plot = df[[parameter, score]].copy()
  
  # Convert param column to int (sometimes it's string/object)
  df_plot[parameter] = df_plot[parameter].astype(int)
  
  # Sort so the plot looks nice
  df_plot = df_plot.sort_values(parameter)

  plt.figure(figsize=(6,4))
  plt.plot(df_plot[parameter], df_plot[score], marker="o")
  plt.xlabel(parameter)
  plt.ylabel(score)
  plt.title("parameter vs score (from RandomizedSearchCV)")
  plt.grid(True)
  plt.show()

plot_parameter_vs_score(rs, "param_knn__n_neighbors", "mean_test_score")

discplay two parameters agains the score:

In [ ]:
def plot_two_parameters_vs_score(rs, param_group_by, second_param, score):
  df = pd.DataFrame(rs.cv_results_)

  for w, sub in df.groupby(param_group_by):
      plt.plot(sub[second_param].astype(int),
              sub[score],
              marker="o",
              label=f"{param_group_by}={w}")

  plt.xlabel(second_param)
  plt.ylabel(score)
  plt.title(f"{second_param} vs {score} by {param_group_by}")
  plt.legend()
  plt.grid(True)
  plt.show()

plot_two_parameters_vs_score(rs, "param_knn__p", "param_knn__n_neighbors", 
                             "mean_test_score")